# Model A – Structural Feature Engineering

**Goal:** Convert each email’s raw text into a set of **hand‑crafted structural
features** that capture phishing indicators like:
- Excessive use of links and email addresses
- Urgency markers (exclamation marks, ALL CAPS)
- Suspicious keywords (“verify”, “click”, “free”, “urgent”, etc.)
- Technical formatting (HTML tags, IP addresses, raw `=` signs often found in
  phishing or malformed headers)

We use the `raw_text` column from `data/preprocessed/processed_data.csv` because
it was intentionally **not** stripped of punctuation or symbols — exactly what
we need for this analysis.

**Output:** A feature matrix saved to `data/model_a/features.csv` ready for
training a classifier.

## 1. Load the Prepared Dataset
We start from the same `processed_data.csv` used for Model B.
The `raw_text` column contains the original subject + body without linguistic
cleaning.

**Note:** we focus on the `raw_text` column since it contains the characters such as `! @ $ ...` Thus we will be able to extract features related to phishing such as number of puctuations and to don't affect the sturcture of links and to be able to compute the number of subdomains.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "preprocessed" / "processed_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} emails")
print(f"Columns: {df.columns.tolist()}")
df[['raw_text', 'label']].head(2)

Loaded 164467 emails
Columns: ['source_file', 'label', 'subject', 'body', 'raw_text', 'cleaned_text']


,raw_text,label
0,"Never agree to be a loser\nBuck up, your troub...",1
1,Befriend Jenna Jameson\nUpgrade your sex and p...,1


## 2. Define the Structural Feature Extractor
We build a function that scans the raw email text and extracts **17 numeric
features** capturing typical phishing signals:

- **Links & emails**: `num_urls`, `num_emails`, `num_subdomains`  
- **Formatting**: `num_html_tags`, `num_ips`  
- **Punctuation / urgency**: `num_exclamations`, `num_questions`, `uppercase_ratio`  
- **Text shape**: `total_words`, `avg_word_length`  
- **Suspicious keywords**: a fixed list of phishing‑prone words  
- **Reply/Forward markers**: `num_replies_forwards` (RE:/FWD:)

All regex patterns are pre‑compiled for speed.

In [ ]:
import re

URL_PATTERN        = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
EMAIL_PATTERN      = re.compile(r'\S+@\S+')
IP_PATTERN         = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
HTML_TAG_PATTERN   = re.compile(r'<\s*(html|a\s|script|div)', re.IGNORECASE)
RE_FWD_PATTERN     = re.compile(r'\b(?:RE|FWD):', re.IGNORECASE)
URL_HOST_PATTERN   = re.compile(r'(?:https?://)?(?:www\.)?([^/\s:]+)', re.IGNORECASE)

SUSPICIOUS_WORDS = {
    'urgent', 'verify', 'click', 'free', 'winner', 'update',
    'password', 'confirm', 'limited', 'offer', 'claim', 'account',
    'suspended', 'security', 'login', 'unlock', 'congratulations'
}

def count_subdomains(url_host: str) -> int:
    """Number of subdomains (e.g. secure.login.example.com -> 2)."""
    parts = url_host.split('.')
    return max(0, len(parts) - 2)

def extract_structural_features(text: str) -> dict:
    text = str(text)
    
    # URL, email, IP, HTML counts
    urls = URL_PATTERN.findall(text)
    num_urls = len(urls)
    num_emails = len(EMAIL_PATTERN.findall(text))
    num_ips = len(IP_PATTERN.findall(text))
    num_html_tags = len(HTML_TAG_PATTERN.findall(text))
    
    # Punctuation counts
    excl = text.count('!')
    question = text.count('?')
    
    # Uppercase ratio
    alpha_chars = [c for c in text if c.isalpha()]
    upper_ratio = sum(1 for c in alpha_chars if c.isupper()) / max(len(alpha_chars), 1)
    
    # Word statistics
    words = text.split()
    total_words = len(words)
    avg_word_len = sum(len(w) for w in words) / max(total_words, 1)
    
    # Suspicious keywords
    text_lower = text.lower()
    suspicious_count = sum(text_lower.count(word) for word in SUSPICIOUS_WORDS)
    
    # Reply/Forward markers
    re_fwd_count = len(RE_FWD_PATTERN.findall(text))
    
    # Subdomain count from all URLs
    subdomain_total = 0
    for url in urls:
        host_match = URL_HOST_PATTERN.search(url)
        if host_match:
            host = host_match.group(1)
            subdomain_total += count_subdomains(host)
    
    return {
        'num_urls': num_urls,
        'num_emails': num_emails,
        'num_ips': num_ips,
        'num_html_tags': num_html_tags,
        'num_exclamations': excl,
        'num_questions': question,
        'uppercase_ratio': upper_ratio,
        'total_words': total_words,
        'avg_word_length': avg_word_len,
        'suspicious_word_count': suspicious_count,
        'num_replies_forwards': re_fwd_count,
        'num_subdomains': subdomain_total,
    }

# Test on the first email
print(extract_structural_features(df['raw_text'].iloc[0]))

{'num_urls': 1, 'num_emails': 0, 'num_ips': 0, 'num_html_tags': 0, 'num_exclamations': 2, 'num_questions': 0, 'uppercase_ratio': 0.030042918454935622, 'total_words': 52, 'avg_word_length': 4.730769230769231, 'suspicious_word_count': 0, 'num_replies_forwards': 0, 'num_subdomains': 0}


## 3. Build the Feature Matrix & Save
We apply `extract_structural_features` to every email's `raw_text`, create a
DataFrame with all numeric features plus the binary `label`, and save it to
`data/model_a/extracted_features/features.csv`.

This file becomes the direct input for Model A’s training notebook.

In [3]:
features_list = df['raw_text'].apply(extract_structural_features).tolist()
X_a = pd.DataFrame(features_list)

X_a['label'] = df['label'].values

# Ensure output directory exists
output_dir = PROJECT_ROOT / "data" / "model_a" / "extracted_features"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "features.csv"
X_a.to_csv(output_path, index=False)

print(f"Feature matrix saved to {output_path}")
print(f"Shape: {X_a.shape}")
print("Feature columns:", X_a.columns.tolist())
X_a.head()

Feature matrix saved to C:\Work\current\phishing_detection\data\model_a\extracted_features\features.csv
Shape: (164467, 13)
Feature columns: ['num_urls', 'num_emails', 'num_ips', 'num_html_tags', 'num_exclamations', 'num_questions', 'uppercase_ratio', 'total_words', 'avg_word_length', 'suspicious_word_count', 'num_replies_forwards', 'num_subdomains', 'label']


,num_urls,num_emails,num_ips,num_html_tags,num_exclamations,num_questions,uppercase_ratio,total_words,avg_word_length,suspicious_word_count,num_replies_forwards,num_subdomains,label
0,1,0,0,0,2,0,0.030043,52,4.730769,0,0,0,1
1,1,0,0,0,0,0,0.047059,12,7.500000,0,0,0,1
2,24,1,0,0,0,15,0.267820,306,11.068627,0,0,1,1
3,652,1,0,0,4,75,0.012676,2670,8.118727,2,1,6,0
4,1,0,0,0,0,0,0.238095,3,65.333333,0,0,2,1
